<a href="https://colab.research.google.com/github/ewarren38/HW8_WarrenE/blob/main/HW8_WarrenE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Author: Eleanor Warren

Class: ST554 (601)

Date: 4/5/26

Purpose: HW8 (and improvement on HW7)

**Updating some work from HW7 using HW7 key, then continuing work on HW8. I made these changes to make it easier for me to complete HW8 correctly.**

Key changes include:

- No longer using `type` as a categorical predictor in any of my models for simplicity.
- Doing my train/test splitting for both types of response variables (numeric and categorical) in one step, instead of repeating later.
- Standardizing all of my training and test data in one step.
- Finalizing my analysis for the logistic regressions.

###**Work from HW7**

Read in the red and white wine quality data, and display the first few values. The delimiter here is a `;`. Adding a "type" variable to both the red and white datasets before concatenating.

In [ ]:
import pandas as pd
red = pd.read_csv("/content/winequality-red.csv", sep = ";")
red["type"] = "red"
red.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,red


In [ ]:
white = pd.read_csv("/content/winequality-white.csv", sep = ";")
white["type"] = "white"
white.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6,white
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6,white
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6,white
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,white
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,white


Combining the datasets into a "full" wine dataset.


In [ ]:
wine = pd.concat([red, white], ignore_index = True)
wine

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6,white
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5,white
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6,white
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7,white


First, we'll want to treat `alcohol` as our response. Because of that, we'll exclude it from our training data. We'll also use `type` as a stratification variable, since we want to ensure that we have a similar proportion of red and white wines in the training and test sets.

I'm also going to drop `type` here since it will be the response variable for our logistic regression. This way I can take care of the train/test splitting all at once and not have to repeat this process. It looks like I can keep quality in here even though it's not our response variable anymore.

In [ ]:
from sklearn.model_selection import train_test_split

# Split our data into a training and testing dataset, stratifying on "type"
X_train, X_test, y_train, y_test = train_test_split(
  wine.drop(["alcohol", "type"], axis = 1), # axis = 1 means you're dropping a column.
  wine[["alcohol", "type"]], # use alcohol as our response
  test_size=0.20, # 80 / 20 split
  random_state=41, # setting a seed
  stratify = wine["type"]) # our stratify column

Before we make our MLR models, we will want to standardize our regressors, since the sulfur metrics are much larger in magnitude than the other variables and we don't want that to artifically cause it to matter more in our model.

We don't want to standardize our **test** set by it's own mean/standard deviation, we want to standardize it by our **training** set mean/standard deviation. That's because when we're assessing our model we don't want anything to rely on the **test** set! As such, we'll find and save the mean and standard deviation of our training data here.

In [ ]:
import numpy as np

# find the mean of all the columns
means = X_train.apply(np.mean, axis = 0)
means

,0
fixed acidity,7.219925
volatile acidity,0.339677
citric acid,0.318339
residual sugar,5.472321
chlorides,0.056219
free sulfur dioxide,30.419473
total sulfur dioxide,115.381278
density,0.994713
pH,3.218232
sulphates,0.531339


In [ ]:
# find the standard deviation of all the columns
std = X_train.apply(np.std, axis = 0)
std

,0
fixed acidity,1.295153
volatile acidity,0.165221
citric acid,0.144098
residual sugar,4.769460
chlorides,0.035877
free sulfur dioxide,17.715183
total sulfur dioxide,55.981301
density,0.003021
pH,0.160119
sulphates,0.150413


Use a lambda function to standardize the numeric variables in my training data. Then create a function we can use to easily standardize the test data using the mean and std metrics from our training dataset.

In [ ]:
# For all the variables in our x training dataset, replace the variable with itself
# minus (elementwise) the mean of itself, divided by the standard deviation of itself
# which we specify by axis = 0 so that these calculations are done up and down the rows
# of a variable.
X_train = X_train.apply(lambda x: (x-np.mean(x))/np.std(x), axis = 0) # axis = 0 means we take in columns

# Quick function to standardize based off of a supplied mean and std
def my_std_fun(x, means, std):
    return(x-means)/std

# Go ahead and standardize my "test" dataset as well using "training" mean/std
# recall, we saved "means" and "std" above from our TRAINING set data, that's what we're using here
for x in X_test.columns:
    X_test[x] = my_std_fun(X_test[x], means[x], std[x])

Do a quick inspection of our standardised training dataset now to make sure the values look smaller/closer together/standardized correctly.

In [ ]:
X_test.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality
2133,0.139038,-0.542769,0.289112,1.620242,0.300504,1.274643,1.118565,0.955790,-1.300484,-0.341322,0.211357
5984,-0.633072,-0.482244,-0.751845,0.571905,-0.173336,1.161745,1.190017,0.234145,0.073498,-0.341322,-0.929266
2728,0.525093,-0.542769,-0.196668,-0.036130,-0.702921,0.145668,0.028915,-0.964182,-0.988216,-1.405062,1.351979
234,0.756725,3.996607,-1.584610,-0.665132,0.244758,-1.322000,-1.400133,0.707517,0.635581,0.124064,0.211357
6093,0.216249,-0.603294,-0.196668,-0.916733,-0.284827,0.202116,-0.560567,-1.493829,-1.113123,-0.939676,0.211357


**MULTIPLE LINEAR REGRESSION MODELS**

Creating a version of our standardized training data that includes some interaction terms.

- `chlor_sulph_int`: is the interaction between chlorides and sulphates
- `pH_square`: is a polynomial term degree 2 for pH

In [ ]:
# Creating an interaction term using a better method:
from sklearn.preprocessing import PolynomialFeatures

# Interaction
inter_poly = PolynomialFeatures(interaction_only=True, include_bias = False) # we only want interaction of the two vars
inter_train = inter_poly.fit_transform(X_train[["chlorides", "sulphates"]]) # since we did 'interaction only = True' this
                                                                            # returns a 2D array with both original vars and
                                                                            # their interaction.

# Quadratic term
poly = PolynomialFeatures(degree = 2, include_bias = False) # we want a quadratic ^2 term
poly_train = poly.fit_transform(X_train[["pH"]])  # returns a 2D array where the first column
                                                  # is the original var and the second column is the square

Quick inspection of the dimensions of our resulting arrays.

In [ ]:
#poly_train.shape
#inter_train

**`MLR1`**

Below we'll fit four different models. The first model is an MLR with the following regressors. This includes everything from our original dataset except for `quality` or any `type` indicator. I use `cross_validate` to find the `neg_mean_squared_error` for each MLR model I build to be able to find the best one.

- `["fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates"]`

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate

# Model with all of the original predictors in it (plus type)
cv_mlr1 = cross_validate(
    LinearRegression(),
    X_train[["fixed acidity", "volatile acidity", "citric acid", "residual sugar", \
             "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density", \
             "pH", "sulphates"]],
    y_train["alcohol"], # specify alcohol since we also have "type" in the response dataset
    cv = 5, # 5 folds cross validation
    scoring = "neg_mean_squared_error") # MSE is our metric for how good it is

**`MLR2`**

My next model is another MLR with a subset of the original variables, so it's simpler fewer linear terms. These are ones I think might be important.

- `["fixed acidity", "residual sugar", "chlorides", "sulphates", "pH"]`

In [ ]:
# Model with a selection of what I think are important, no polynomial or interaction
cv_mlr2= cross_validate(
    LinearRegression(),
    X_train[["fixed acidity", "residual sugar", "chlorides", "sulphates", "pH"]],
    y_train["alcohol"],
    cv = 5,
    scoring = "neg_mean_squared_error")

**`MLR3`**

This model includes the same variables as `MLR2`, but with a quadratic term for `pH`. First, I have to make a new array of regressors for this model since I want to include the quadratic `pH` term.

- `["fixed acidity", "residual sugar", "chlorides", "sulphates", "pH", "pH_square"]`

In [ ]:
# Include our polynomial term in the array of regressors we want to use.
X_array = X_train[["fixed acidity", "residual sugar", "chlorides", \
             "sulphates", "pH"]].copy() # make a copy instead of linking to original X_train
X_array["pH_square"] = poly_train[:,1]  # pull the pH_square column only
X_array.head()

#quad_term = poly_train[:,1].reshape(-1,1)

,fixed acidity,residual sugar,chlorides,sulphates,pH,pH_square
4185,-0.324228,1.536375,0.161139,-0.075387,-1.362938,1.857600
3599,-0.324228,-0.853833,-0.452065,-0.474290,-0.176317,0.031088
4631,-0.401439,-0.811899,-0.563556,-0.274839,0.947850,0.898419
4009,-0.169806,0.383205,-0.256954,0.323515,1.010303,1.020713
2794,0.525093,1.829910,-0.507811,-0.474290,-1.175577,1.381981


In [ ]:
# Model with a polynomial term for pH
cv_mlr3 = cross_validate(
    LinearRegression(),
    X_array,
    y_train["alcohol"],
    cv = 5,
    scoring = "neg_mean_squared_error")

**`MLR4`**

This model includes the same variables as `MLR2`, but this time with the interaction term between `chlorides` and `sulphates`. First, I have to make a new array of regressors for this model since I want to include the quadratic `pH` term.

- `["fixed acidity", "residual sugar", "chlorides", "sulphates", "pH", "chlor_sulph_int"]`

In [ ]:
# Include our polynomial term in the array of regressors we want to use.
X_array_int = X_train[["fixed acidity", "residual sugar", "chlorides", \
             "sulphates", "pH"]].copy() # make a copy instead of linking to original X_train
X_array_int["chlor_sulph_int"] = inter_train[:,2]  # pull the interaction column only
X_array_int.head()

,fixed acidity,residual sugar,chlorides,sulphates,pH,chlor_sulph_int
4185,-0.324228,1.536375,0.161139,-0.075387,-1.362938,-0.012148
3599,-0.324228,-0.853833,-0.452065,-0.474290,-0.176317,0.214410
4631,-0.401439,-0.811899,-0.563556,-0.274839,0.947850,0.154887
4009,-0.169806,0.383205,-0.256954,0.323515,1.010303,-0.083129
2794,0.525093,1.829910,-0.507811,-0.474290,-1.175577,0.240849


In [ ]:
# Model with an interaction term for chlorides and sulfates
cv_mlr4 = cross_validate(
    LinearRegression(),
    X_array_int, # these variables in our model
    y_train["alcohol"],
    cv = 5,
    scoring = "neg_mean_squared_error")

**BEST MULTIPLE LINEAR REGRESSION MODEL**

Find the root MSE for each of the MLR models. We will choose the model with the smallest value. `MLR1` is the best here, with an RMSE of `1.245`



In [ ]:
print(np.sqrt(-sum(cv_mlr1['test_score'])),
      np.sqrt(-sum(cv_mlr2['test_score'])),
      np.sqrt(-sum(cv_mlr3['test_score'])),
      np.sqrt(-sum(cv_mlr4['test_score'])))

1.2448542096064572 2.3411391818503167 2.340747265737467 2.3120346263217666


`MLR1` had the lowest MSE. Fit our best model on all of our training data now (not just each individual fold).

In [ ]:
mlr_best = LinearRegression().fit(X_train[["fixed acidity", "volatile acidity", \
                                           "citric acid", "residual sugar", \
                                           "chlorides", "free sulfur dioxide", \
                                           "total sulfur dioxide", "density", \
                                           "pH", "sulphates"]], y_train["alcohol"])
print(mlr_best.intercept_)
print(np.array(list(zip(X_train.columns, mlr_best.coef_))))

10.489225835417816
[['fixed acidity' '0.7237624011665407']
 ['volatile acidity' '0.2357279468917417']
 ['citric acid' '0.058701627760184755']
 ['residual sugar' '0.9939489120313405']
 ['chlorides' '-0.008680057339563618']
 ['free sulfur dioxide' '0.011479724763599208']
 ['total sulfur dioxide' '-0.268060546379285']
 ['density' '-1.8302098403560705']
 ['pH' '0.46976417764535217']
 ['sulphates' '0.2172302355557649']]


**LASSO MODEL**

Next, do a LASSO model with at least 5 predictors. I chose a subset of the same ones from my best MLR, removing some I think are redundant. The LASSO model will find our optimal alpha for the penalty.

- `[["fixed acidity", "citric acid", "residual sugar", "chlorides", "total sulfur dioxide", "pH", "sulphates"]]`

In [ ]:
from sklearn.linear_model import LassoCV, Lasso

# we aren't specifying our alphas, we're going to let LASSO find it for us
# 5 cross-validation folds
lasso_mod = LassoCV(cv=5, random_state=0) \
    .fit(X_train[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates"]],
         y_train["alcohol"])

As we did in class, we can inspect the alphas that the LASSO model found to identify the one associated with the lowest MSE.

In [ ]:
# take the array of 100 alphas and the corresponding MSE's and zip them together
# across columns, then make a list of those two-pair lists and print in descending
# order according to MSE to figure out which alpha yields the lowest MSE.

np.set_printoptions(suppress = True)
fit_info = np.array(list(zip(lasso_mod.alphas_, np.mean(lasso_mod.mse_path_, axis = 1))))
fit_info.shape # there are 100 rows, and 2 columns

# this means that across all 100 rows (of dim1), we
# look at the second column (index 1) of dim2, and sort all the arguments
# and that gives us the indexes in sorted order?
fit_info[:,1].argsort()

# lastly, we want to print off the first 10 rows, across all columns, rounded to 4
# of our array in the index order specified from the sort.
fit_info[fit_info[:,1].argsort()][0:10,].round(4)

array([[0.0011, 1.0087],
       [0.001 , 1.0087],
       [0.0009, 1.0087],
       [0.0009, 1.0087],
       [0.0008, 1.0087],
       [0.0012, 1.0087],
       [0.0008, 1.0087],
       [0.0007, 1.0087],
       [0.0012, 1.0087],
       [0.0007, 1.0087]])

Now we have the alpha, intercept, and coefficients for our best LASSO model. Notice that none of the variables were set equal to 0, which could've happened since LASSO helps us do model selection. That's a good sign that we chose informative columns.

In [ ]:
print(lasso_mod.alpha_)
print(lasso_mod.intercept_)
print(np.array(list(zip(X_train.columns, lasso_mod.coef_))))

0.0010740226880774238
10.48922583541787
[['fixed acidity' '-0.23970712944635306']
 ['volatile acidity' '0.1789789027080025']
 ['citric acid' '-0.35913556156586396']
 ['residual sugar' '-0.40164477842460095']
 ['chlorides' '-0.36625373531683153']
 ['free sulfur dioxide' '-0.03503701708947504']
 ['total sulfur dioxide' '0.04595621844378307']]


Okay, so now we have our best LASSO model, and we fit it across the entire training dataset (not just one fold at a time)

In [ ]:
lasso_best = Lasso(lasso_mod.alpha_) \
                  .fit(X_train[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates"]], y_train["alcohol"])

**RIDGE REGRESSION**

We also want to fit our model using RidgeCV. Below we use the same regressors as the LASSO model.

- `[["fixed acidity", "citric acid", "residual sugar", "chlorides", "total sulfur dioxide", "pH", "sulphates"]]`

In [ ]:
from sklearn.linear_model import Ridge, RidgeCV
ridge_mod = RidgeCV(cv = 5)
ridge_mod.fit(X_train[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates"]], y_train["alcohol"])

RidgeCV(cv=5)

Use the best alpha that the RidgeCV found for us, and refit the model on all of our training data.

In [ ]:
ridge = Ridge(alpha = ridge_mod.alpha_)
ridge_best = ridge.fit(X_train[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates"]], y_train["alcohol"])

**ELASTIC NET**

Using the same regressors as the LASSO and Ridge models. We have to pass some `l1_ratio` and `alpha` values for the `ElasticNetCV` model to try, and determine which produces the optimal score.

- `[["fixed acidity", "citric acid", "residual sugar", "chlorides", "total sulfur dioxide", "pH", "sulphates"]]`

In [ ]:
from sklearn.linear_model import ElasticNet, ElasticNetCV

# Pass through l1_ratio and alphas
elastic_mod = ElasticNetCV(cv=5,
                    random_state=0,
                    l1_ratio = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.96, 0.98, 0.99, 1],
                    n_alphas = 300)
elastic_mod.fit(X_train[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates"]], y_train["alcohol"])


ElasticNetCV(cv=5,
             l1_ratio=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.96, 0.98, 0.99, 1],
             n_alphas=300, random_state=0)

Use the best `alpha` and `l1_ratio` and refit our model on all of our training data.

In [ ]:
en = ElasticNet(alpha = elastic_mod.alpha_, l1_ratio = elastic_mod.l1_ratio_)
elastic_best = en.fit(X_train[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates"]], y_train["alcohol"])

**PREDICTIONS**

Update: Since I've corrected the work from HW7 to already have standardized my test dataset too, I can go ahead and run my predictions using my best models of each class.



In [ ]:
mlr_pred = mlr_best.predict(X_test[["fixed acidity", "volatile acidity", \
                                    "citric acid", "residual sugar", \
                                    "chlorides", "free sulfur dioxide", \
                                    "total sulfur dioxide", "density", "pH", \
                                    "sulphates"]])

In [ ]:
lasso_pred = lasso_best.predict(X_test[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates"]])

In [ ]:
ridge_pred = ridge_best.predict(X_test[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates"]])

In [ ]:
en_pred = elastic_best.predict(X_test[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates"]])

**CHOOSE MODEL**

Lastly, we want to find the MSE between the true y test values and our predicted values from the different models. We see that the MLR model does the best job here with an RMSE of `0.5299`. We'll choose the MLR model!

In [ ]:
from sklearn.metrics import mean_squared_error

print([np.sqrt(mean_squared_error(y_test["alcohol"], mlr_pred)),
       np.sqrt(mean_squared_error(y_test["alcohol"], lasso_pred)),
       np.sqrt(mean_squared_error(y_test["alcohol"], ridge_pred)),
       np.sqrt(mean_squared_error(y_test["alcohol"], en_pred))])

[np.float64(0.5299164662668598), np.float64(1.02337230099641), np.float64(1.023378114453307), np.float64(1.023411859149173)]


**LOGISTIC REGRESSION (BINARY RESPONSE)**

For this next section, our response variable is going to be binary instead of continuous.

**LG1**

- This model includes the same longer list of regressors as our optimal MLR model: `["fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates"]`
- The penalty is `None`
- The scoring is based on negative log loss

In [ ]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

# Create an instance of our Logistic Regression
cv_lg1 = cross_validate(LogisticRegression(penalty = None),
      X_train[["fixed acidity", "volatile acidity", "citric acid", \
             "residual sugar", "chlorides", "free sulfur dioxide", \
             "total sulfur dioxide", "density", "pH", "sulphates"]],
      y_train["type"],
      cv = 5,
      scoring = "neg_log_loss")

**LG2**

- This model includes a subset of the first model: `["fixed acidity", "residual sugar", "chlorides", "sulphates", "pH"]`
- The penalty is `None`
- The scoring is based on negative log loss

In [ ]:
cv_lg2 = cross_validate(LogisticRegression(penalty = None),
      X_train[["fixed acidity", "residual sugar", "chlorides", \
             "sulphates", "pH"]],
      y = y_train["type"],
      cv = 5,
      scoring = "neg_log_loss")

**LG3**

I want to include my quadratic term for `pH` here, so I use my dataset with that variable included from before, `X_array`

In [ ]:
cv_lg3 = cross_validate(LogisticRegression(penalty = None),
      X_array[["fixed acidity", "residual sugar", "chlorides", \
             "sulphates", "pH", "pH_square"]],
      y = y_train["type"],
      cv = 5,
      scoring = "neg_log_loss")

**LG4**

Replace the quadratic `pH` term with the interaction term, `chlor_sulph_int`. I use the dataset with the interaction term, `X_array_int`

In [ ]:
cv_lg4 = cross_validate(LogisticRegression(penalty = None),
      X_array_int[["fixed acidity", "residual sugar", "chlorides", \
             "sulphates", "chlor_sulph_int", "pH"]],
      y = y_train["type"],
      cv = 5,
      scoring = "neg_log_loss")

**BEST LOGISTIC REGRESSION MODEL**

According to the HW7 key, the values that are _closer to zero_ are better, so we want the _larger_ value when they are all negative. Here, that would make the first model the best, with a score of `-0.2008`.

In [ ]:
[round(cv_lg1['test_score'].sum(),4), round(cv_lg2['test_score'].sum(),4), round(cv_lg3['test_score'].sum(),4), round(cv_lg4['test_score'].sum(),4)]

[np.float64(-0.2008),
 np.float64(-0.7535),
 np.float64(-0.7255),
 np.float64(-0.7258)]

Fit the best Logisitic Regression Model:

In [ ]:
# Fit the best LG1 model
lg1_best = LogisticRegression(penalty = None, solver = "newton-cholesky") \
                .fit(X_train[["fixed acidity", "volatile acidity", "citric acid", \
                "residual sugar", "chlorides", "free sulfur dioxide", \
                "total sulfur dioxide", "density", "pH", "sulphates"]],
                y_train["type"])

**LASSO**

Now we create our LASSO model for a binary response.

- `["fixed acidity", "citric acid", "residual sugar", "chlorides", "total sulfur dioxide", "pH", "sulphates"]`



In [ ]:
# LASSO
lg_lasso = LogisticRegressionCV(cv = 5,
                               solver = "saga",
                               penalty = "l1",
                               Cs = 200,
                               max_iter = 1000,
                               scoring = "neg_log_loss",
                               random_state = 10)
lg_lasso.fit(X_train[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates"]], y_train["type"])

lg_lasso.C_

array([0.87035914])

Fit our best LASSO model

In [ ]:
lg_lasso_best = LogisticRegression(
    solver = "saga",
    penalty = "l1",
    C = lg_lasso.C_[0],
    random_state = 10) \
   .fit(X_train[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates"]], y_train["type"])

**RIDGE**

- `["fixed acidity", "citric acid", "residual sugar", "chlorides", "total sulfur dioxide", "pH", "sulphates"]`

In [ ]:
# Ridge
lg_ridge = LogisticRegressionCV(cv = 5,
                               solver = "newton-cg",
                               penalty = "l2",
                               Cs = 200,
                               max_iter = 1000,
                               scoring = "neg_log_loss",
                               random_state = 10)
lg_ridge.fit(X_train[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates"]], y_train["type"])

lg_ridge.C_

array([1.51671689])

Fit best Ridge model

In [ ]:
lg_ridge_best = LogisticRegression(
    solver = "newton-cg",
    penalty = "l2",
    C = lg_ridge.C_[0],
    random_state = 10) \
   .fit(X_train[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates"]],
         y_train["type"])

**ELASTIC NET**

- `["fixed acidity", "citric acid", "residual sugar", "chlorides", "total sulfur dioxide", "pH", "sulphates"]`
- Again, we supply some `l1_ratio` values for the CV model to use and find the best one.

In [ ]:
# ElasticNet
lg_en = LogisticRegressionCV(cv = 5,
                               solver = "saga",
                               penalty = "elasticnet",
                               Cs = 200,
                               l1_ratios = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.96, 0.98, 0.99, 1],
                               scoring = "neg_log_loss",
                               max_iter = 1000,
                               random_state = 10)
lg_en.fit(X_train[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates"]], y_train["type"])

lg_en.C_

array([1.26038293])

Fit best Elastic Net model

In [ ]:
lg_en_best = LogisticRegression(
    solver = "saga",
    penalty = "elasticnet",
    C = lg_en.C_[0],
    l1_ratio = lg_en.l1_ratio_[0],
    random_state = 10) \
    .fit(X_train[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", \
                  "total sulfur dioxide", "pH", \
                  "sulphates"]],
         y_train["type"])

**NOTE**

I've updated my work thus far to adhere better to the HW7 answer key to make my work for HW8 easier.

**COMPARE MODELS**

First, compare models based on their prediction accuracy, which is how many observations they correctly categorized out of the total number.

In [ ]:
from sklearn.metrics import  accuracy_score

# Logistic Regression
lg1_pred = lg1_best.predict(X_test[["fixed acidity", "volatile acidity", \
                "citric acid", "residual sugar", "chlorides", "free sulfur dioxide", \
                "total sulfur dioxide", "density", "pH", "sulphates"]])
print(accuracy_score(y_test["type"], lg1_pred))

# Logistic LASSO
lg_lasso_pred = lg_lasso_best.predict(X_test[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", "total sulfur dioxide", "pH", \
                  "sulphates"]])
print(accuracy_score(y_test["type"], lg_lasso_pred))

# Logistic Ridge
lg_ridge_pred = lg_ridge_best.predict(X_test[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", "total sulfur dioxide", "pH", \
                  "sulphates"]])
print(accuracy_score(y_test["type"], lg_ridge_pred))

# Logistic Elastic Net
lg_en_pred = lg_en_best.predict(X_test[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", "total sulfur dioxide", "pH", \
                  "sulphates"]])
print(accuracy_score(y_test["type"], lg_en_pred))

0.9930769230769231
0.9807692307692307
0.9807692307692307
0.9807692307692307


It's clear that the first Logistic Regression model `LG1` has the highest accuracy score. Do the same thing, but with log loss as our model metric. Here, the smallest value is the best. Again, our first model `LG1` performs the best, with a log_loss of `0.487`



In [ ]:
from sklearn.metrics import log_loss

lg1_pred_log = lg1_best.predict_proba(X_test[["fixed acidity", "volatile acidity", \
                "citric acid", "residual sugar", "chlorides", "free sulfur dioxide", \
                "total sulfur dioxide", "density", "pH", "sulphates"]])
print(log_loss(y_test["type"], lg1_pred_log))

# Logistic LASSO
lg_lasso_pred_log = lg_lasso_best.predict_proba(X_test[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", "total sulfur dioxide", "pH", \
                  "sulphates"]])
print(log_loss(y_test["type"], lg_lasso_pred_log))

# Logistic Ridge
lg_ridge_pred_log = lg_ridge_best.predict_proba(X_test[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", "total sulfur dioxide", "pH", \
                  "sulphates"]])
print(log_loss(y_test["type"], lg_ridge_pred_log))

# Logistic Elastic Net
lg_en_pred_log = lg_en_best.predict_proba(X_test[["fixed acidity", "citric acid", \
                  "residual sugar", "chlorides", "total sulfur dioxide", "pH", \
                  "sulphates"]])
print(log_loss(y_test["type"], lg_en_pred_log))

0.048650992146656885
0.07028295364943929
0.07035985291950429
0.0703385435724148


###**HW8**

Fit a regression tree with `alcohol` as a response.

- Include 5 predictors
- Use CV to select the tuning parameters, `max_depth` and `min_samples_leaf`

Here, I'm going to use `["fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates", "quality"]`

This time I'm including `quality`, whereas I'd excluded it from `HW7`

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

parameters_tree = {'max_depth': range(2, 15),
              'min_samples_leaf': [3, 5, 10, 50, 100]}

tuning_tree = GridSearchCV(DecisionTreeRegressor(),
                            parameters_tree,
                            cv = 5,
                            scoring = "neg_mean_squared_error")
tuning_tree.fit(X_train[["fixed acidity", "volatile acidity", "citric acid", \
                       "residual sugar", "chlorides", "free sulfur dioxide", \
                       "total sulfur dioxide", "density", "pH", "sulphates", \
                       "quality"]], y_train["alcohol"])

GridSearchCV(cv=5, estimator=DecisionTreeRegressor(),
             param_grid={'max_depth': range(2, 15),
                         'min_samples_leaf': [3, 5, 10, 50, 100]},
             scoring='neg_mean_squared_error')

Inspect the best tuning parameters

In [ ]:
print(tuning_tree.best_estimator_)

DecisionTreeRegressor(max_depth=10, min_samples_leaf=10)


In [ ]:
print(tuning_tree.best_score_, tuning_tree.best_params_)

-0.45719189958730533 {'max_depth': 10, 'min_samples_leaf': 10}


Refit the Regression Tree model with its best parameters on the full training set, not just the 5 folds.

In [ ]:
tree_best = DecisionTreeRegressor(max_depth = 10,
                                  min_samples_leaf = 10,
                                  random_state = 21)
tree_best.fit(X_train[["fixed acidity", "volatile acidity", "citric acid", \
                       "residual sugar", "chlorides", "free sulfur dioxide", \
                       "total sulfur dioxide", "density", "pH", "sulphates", \
                       "quality"]], y_train["alcohol"])

DecisionTreeRegressor(max_depth=10, min_samples_leaf=10, random_state=21)

Now we want to fit a Random Forest model

In [ ]:
from sklearn.ensemble import RandomForestRegressor

parameters_forest = {"max_features": range(1,4), \
                     "max_depth": [3, 4, 5, 10, 15],\
                     'min_samples_leaf':[3, 10, 50, 100]}
tuning_forest = GridSearchCV(RandomForestRegressor(n_estimators = 500),
                       parameters_forest, \
                       cv = 5, \
                       scoring = "neg_mean_squared_error")
tuning_forest.fit(X_train[["fixed acidity", "volatile acidity", "citric acid", \
                       "residual sugar", "chlorides", "free sulfur dioxide", \
                       "total sulfur dioxide", "density", "pH", "sulphates", \
                       "quality"]], y_train["alcohol"])

GridSearchCV(cv=5, estimator=RandomForestRegressor(n_estimators=500),
             param_grid={'max_depth': [3, 4, 5, 10, 15],
                         'max_features': range(1, 4),
                         'min_samples_leaf': [3, 10, 50, 100]},
             scoring='neg_mean_squared_error')

In [ ]:
print(tuning_forest.best_score_, tuning_forest.best_params_, tuning_forest.best_estimator_)

-0.3445446648847054 {'max_depth': 15, 'max_features': 3, 'min_samples_leaf': 3} RandomForestRegressor(max_depth=15, max_features=3, min_samples_leaf=3,
                      n_estimators=500)


Refit the model with the full training dataset and the best parameters

In [ ]:
forest_best = RandomForestRegressor(n_estimators = 500,
                                  max_depth = 15,
                                  max_features = 3,
                                  min_samples_leaf = 3,
                                  random_state = 21)
forest_best.fit(X_train[["fixed acidity", "volatile acidity", "citric acid", \
                       "residual sugar", "chlorides", "free sulfur dioxide", \
                       "total sulfur dioxide", "density", "pH", "sulphates", \
                       "quality"]], y_train["alcohol"])

RandomForestRegressor(max_depth=15, max_features=3, min_samples_leaf=3,
                      n_estimators=500, random_state=21)

See how they perform on the test set.

